# CanastaCL — 02 · Limpieza y normalización

**Contexto desde el EDA:** el dataset crudo no tiene nulos, duplicados ni incoherencias de precio. La limpieza se concentra en:

1. **Normalización de columnas** a `snake_case`.
2. **Normalización de unidades** — 19 unidades distintas reducidas a 3 categorías (peso, volumen, unidad) con factores de conversión a `$/kilo`, `$/litro` y `$/unidad individual`.
3. **Features temporales** (`semana_iso`, `mes_num`, `dia_anio`).
4. **ID de producto estable** (`producto + unidad + tipo_punto`) para comparar series sin mezclar canales.
5. **Persistencia** en `data/processed/precios_clean.parquet`.

Toda la lógica vive en `src/features.py` para mantener el notebook delgado y reutilizar desde el dashboard y los modelos.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.data_loader import load_raw, PROCESSED_DIR
from src.features import (
    UNIDAD_MAP,
    build_clean_dataset,
)

pd.set_option("display.max_columns", 50)

## 2. Carga del dataset crudo

In [ ]:
df_raw = load_raw()
print(f"Filas: {len(df_raw):,} | Columnas: {df_raw.shape[1]}")
df_raw.head(3)

## 3. Validaciones de calidad (re-confirmación post-EDA)

In [ ]:
checks = {
    "filas": len(df_raw),
    "nulos_total": int(df_raw.isna().sum().sum()),
    "duplicados_exactos": int(df_raw.duplicated().sum()),
    "precio_min_negativo": int((df_raw["Precio minimo"] < 0).sum()),
    "precio_prom_negativo": int((df_raw["Precio promedio"] < 0).sum()),
    "precio_min_mayor_max": int((df_raw["Precio minimo"] > df_raw["Precio maximo"]).sum()),
    "precio_prom_fuera_rango": int(
        ((df_raw["Precio promedio"] < df_raw["Precio minimo"]) | (df_raw["Precio promedio"] > df_raw["Precio maximo"])).sum()
    ),
}
pd.Series(checks, name="valor")

In [ ]:
# Si en el futuro se cargan más años o fuentes y aparecen filas inválidas,
# este filtro las elimina conservando un log mínimo.
antes = len(df_raw)
df_raw = df_raw.dropna(subset=["Precio promedio", "Producto", "Unidad"])
df_raw = df_raw[df_raw["Precio promedio"] > 0]
df_raw = df_raw[df_raw["Precio minimo"] <= df_raw["Precio maximo"]]
df_raw = df_raw[(df_raw["Precio promedio"] >= df_raw["Precio minimo"]) & (df_raw["Precio promedio"] <= df_raw["Precio maximo"])]
print(f"Filas eliminadas por reglas de calidad: {antes - len(df_raw)}")

## 4. Mapa de unidades disponibles

Tres categorías con factor de conversión a la base estándar:
- **peso** → `$/kilo`
- **volumen** → `$/litro`
- **unidad** → `$/unidad individual`

Las unidades fuera del mapa quedan con `categoria_unidad = NaN` y no reciben precio normalizado.

In [ ]:
mapa_df = (
    pd.DataFrame(
        [(u, c, f) for u, (c, f) in UNIDAD_MAP.items()],
        columns=["unidad", "categoria", "factor_norm"],
    )
    .sort_values(["categoria", "unidad"])
    .reset_index(drop=True)
)
mapa_df

In [ ]:
# Cobertura: ¿qué porcentaje de filas del crudo cae dentro del mapa?
cobertura = df_raw["Unidad"].isin(UNIDAD_MAP).mean()
print(f"Cobertura del mapa de unidades: {cobertura:.1%}")

fuera_de_mapa = df_raw.loc[~df_raw["Unidad"].isin(UNIDAD_MAP), "Unidad"].value_counts()
if not fuera_de_mapa.empty:
    print("Unidades sin mapeo:")
    print(fuera_de_mapa)
else:
    print("Todas las unidades del crudo están mapeadas.")

## 5. Pipeline de limpieza

`build_clean_dataset` aplica en orden: rename → categoría unidad → precios normalizados → features temporales → producto_id.

In [ ]:
df_clean = build_clean_dataset(df_raw)
print("Shape:", df_clean.shape)
print("\nColumnas finales:")
print(list(df_clean.columns))
df_clean.head(3)

In [ ]:
# Sanity check: precios normalizados por categoría
df_clean.groupby("categoria_unidad", observed=True)[["precio_kilo", "precio_litro", "precio_unitario"]].agg(["count", "mean"]).round(0)

In [ ]:
# Verificación rápida: precio_kilo del Pan (250g) debería ser ~4x el precio_prom
muestra_pan = (
    df_clean[df_clean["unidad"] == "$/pan de 250 gramos"]
    [["producto", "unidad", "precio_prom", "factor_norm", "precio_kilo"]]
    .head(5)
)
muestra_pan

## 6. Persistencia

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "precios_clean.parquet"

df_clean.to_parquet(out_path, index=False, compression="snappy")
print(f"Escrito: {out_path}")
print(f"Tamaño: {out_path.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# Verificar lectura del parquet
df_check = pd.read_parquet(out_path)
print("Shape al releer:", df_check.shape)
df_check.dtypes

## 7. Hallazgos de la limpieza

- ✅ **Calidad confirmada:** 0 nulos, 0 duplicados, 0 incoherencias de precio en el dataset original.
- ✅ **Cobertura de unidades 100%** — las 19 unidades del crudo fueron mapeadas a 3 categorías (peso, volumen, unidad).
- ✅ **Comparabilidad ganada:** ahora `precio_kilo` permite comparar productos del rubro peso entre sí; `precio_unitario` para huevos, panes y bandejas.
- ✅ **`producto_id`** evita mezclar series de un mismo producto vendido en distintos canales (supermercado vs feria).

**Próximo notebook (`03_modelo.ipynb`):** baseline de forecasting sobre productos volátiles identificados en el EDA (huevos, arándanos), comparando ARIMA / Prophet / GBM con features temporales.